# LoadPilot — live demo on Google Colab

An LLM dispatch analyst for SuperDispatch-style auto-transport load boards.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

Cell 1 clones the repo; the model (~6 GB) downloads on the first demo run and is then cached for the session.

In [ ]:
!nvidia-smi

## 1. Get the code

Clones the public repository into `/content/LLM`. If you uploaded the folder manually, skip this cell and `%cd` into it instead.

In [ ]:
REPO_URL = "https://github.com/Tadomeka2/LLM.git"
REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
assert os.path.isdir(REPO_DIR), f"Clone failed: {REPO_DIR} does not exist"
%cd {REPO_DIR}
!git log --oneline -5

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Verify the deterministic tool layer (no model needed)

This should print `All tool smoke tests passed.`

In [ ]:
!python scripts/tool_smoke.py | tail -n 25

## 4. One-shot demo with the local LLM

Watch the `[1] tool ...` lines to see the model choose a tool, then read the final answer.

The first run downloads ~6 GB of weights. On a 16 GB T4 the model loads in fp16, so `--4bit` is **not** needed; add it only for 7B.

In [ ]:
!python scripts/demo.py \
  --model Qwen/Qwen2.5-3B-Instruct \
  --ask "I am in Dallas, TX with an open trailer. Which Texas to Georgia load pays best after fuel and deadhead? Diesel is $3.80." \
  --show-trace

## 5. Multi-step demo: rank → add to watchlist

In [ ]:
!python scripts/demo.py \
  --model Qwen/Qwen2.5-3B-Instruct \
  --ask "Find the best open-trailer load from Texas to Georgia for a driver in Dallas, then add it to my watchlist." \
  --show-trace

## 6. Use the Python API directly (best for a live Q&A in the notebook)

The model stays loaded between questions.

In [ ]:
from loadpilot import Agent, LocalLLM

llm = LocalLLM("Qwen/Qwen2.5-3B-Instruct", verbose=True)
agent = Agent(llm, verbose=True)

for question in [
    "Find open loads from Texas to Georgia after 2026-11-05.",
    "Calculate profit for SD-1004 with 120 deadhead miles.",
    "Are these live SuperDispatch loads?",
]:
    result = agent.run(question)
    print("\nQ:", question)
    print("A:", result.answer)
    print("tools:", result.tools_used)

## 7. Reproduce the evaluation table

This runs all 18 cases and rewrites `docs/EVALUATION.md` and `docs/evaluation_results.json`.
Expect several minutes on a T4.

In [ ]:
!python scripts/evaluate.py --model Qwen/Qwen2.5-3B-Instruct

## Optional: 7B model in 4-bit

A 7B model in 4-bit needs roughly 5 GB of VRAM and is slower, but usually follows the JSON protocol better.
```bash
!python scripts/demo.py --model Qwen/Qwen2.5-7B-Instruct --4bit --ask "..."
```